In [1]:
all_entities = ['Contraindication', 'Eq-Value', 'Severity', 'Drug', 'Observation-Name', 'Age', 'Location', 'Organism-Name', 'Encounter', 'Drug-Name', 'Ethnicity', 'Modifier', 'Condition', 'Eq-Unit', 'Eq-Temporal-Unit', 'Family-Member', 'Organism', 'Other', 'Immunization-Name', 'Polarity', 'Condition-Type', 'Immunization', 'Eq-Operator', 'Eq-Temporal-Recency', 'Procedure-Name', 'Indication', 'Exception', 'Study', 'Language', 'Coreference', 'Provider', 'Acuteness', 'Life-Stage-And-Gender', 'Procedure', 'Risk', 'Death', 'Assertion', 'Allergy-Name', 'Specimen', 'Negation', 'Code', 'Stability', 'Birth', 'Criteria-Count', 'Eq-Comparison', 'Condition-Name', 'Insurance', 'Observation', 'Allergy', 'Eq-Temporal-Period']

In [2]:
import json
from typing import Dict, List, Any
from fhir.resources.patient import Patient
from fhir.resources.procedure import Procedure

In [3]:
with open("data/input.json", "r", encoding="utf-8") as file:
    json_data = json.load(file)

In [15]:
def create_characteristic(description, exclude=False):
    return {
        "description": description,
        "exclude": exclude
    }

def create_definition_by_combination(code, characteristics):
    return {
        "definitionByCombination": {
            "code": code,
            "characteristic": characteristics
        }
    }

def process_node(node):
    if isinstance(node, dict):
        if "AND" in node:
            left = process_node(node["AND"]["left"])
            right = process_node(node["AND"]["right"])
            return create_definition_by_combination("all-of", [left, right])
        elif "OR" in node:
            left = process_node(node["OR"]["left"])
            right = process_node(node["OR"]["right"])
            return create_definition_by_combination("any-of", [left, right])
        elif "NOT" in node:
            inner = process_node(node["NOT"]["left"])
            inner["exclude"] = True
            return inner
        else:
            description = node.get("raw_text", "")
            characteristic = create_characteristic(description)

            for key, value in node.items():
                if key not in ["raw_text", "Eq-Operator"]:
                    characteristic[key] = value

            return characteristic
    return None

def convert_json_to_fhir(input_json):
    fhir_structure = process_node(input_json)

    return {
        "resourceType": "EvidenceVariable",
        "status": "draft",
        "characteristic": [fhir_structure]
    }
fhir_output = convert_json_to_fhir(json_data)
print(json.dumps(fhir_output, indent=2))

{
  "resourceType": "EvidenceVariable",
  "status": "draft",
  "characteristic": [
    {
      "definitionByCombination": {
        "code": "all-of",
        "characteristic": [
          {
            "definitionByCombination": {
              "code": "all-of",
              "characteristic": [
                {
                  "definitionByCombination": {
                    "code": "all-of",
                    "characteristic": [
                      {
                        "description": "- Age 8yr-18yrs",
                        "exclude": false,
                        "Age": [
                          "Age"
                        ],
                        "Eq-Comparison": [
                          "8yr-18yrs"
                        ],
                        "Eq-Value": [
                          "8",
                          "18"
                        ],
                        "Eq-Temporal-Unit": [
                          "yr",
                          "yrs"